## Importing libraries

In [2]:
from pathlib import Path as Path
import pandas as pd
import numpy as np

## Loading dataset

In [19]:
data_path = Path(
    "/Users/rohankhanna/Documents/track3_agritech_dataset_files"
)

df_arrivals = pd.read_csv(
    data_path / "track3_mandi_arrivals.csv"
)
df_master_clean = pd.read_csv(
    "/Users/rohankhanna/Documents/datathon/cleaned_data/"
    "mandi_master_cleaned.csv",
    dtype={"mandi_id": "string"},
    na_values=["NA"]
)

## Cleaning dataset

In [7]:
# Keep extra duplicate rows for our audit trail.
arrivals_duplicate_log = df_arrivals.loc[
    df_arrivals.duplicated(keep="first")
].copy()

# Create a working copy with exact duplicates removed.
df_arrivals_clean = (
    df_arrivals
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)

print("Original rows:", len(df_arrivals))
print("Duplicates removed:", len(arrivals_duplicate_log))
print("Remaining rows:", len(df_arrivals_clean))

Original rows: 25750
Duplicates removed: 750
Remaining rows: 25000


## Inspecting rows and coloumns

In [8]:
print("Crop names:")
display(
    df_arrivals_clean["crop_name"]
    .value_counts(dropna=False)
    .to_frame("row_count")
)

print("\nQuantity units:")
display(
    df_arrivals_clean["unit"]
    .value_counts(dropna=False)
    .to_frame("row_count")
)

print("\nSample quantities and units:")
display(
    df_arrivals_clean[
        ["arrival_quantity", "unit"]
    ].head(20)
)

Crop names:


,row_count
crop_name,
Mustard,892
Sugarcane,891
Cotton,881
गन्ना,875
Sarson,859
कपास,847
Sarso,842
sugarcane,826
सरसों,825



Quantity units:


,row_count
unit,
NaN,5004
Qtl,2059
KG,1816
Q,1567
quintal,1518
qtl,1485
Quintals,1447
Kilo,1307
KGS,1299



Sample quantities and units:


,arrival_quantity,unit
0,44.841,T
1,332.54,Qtl
2,415.88 qtl,NaN
3,-107.13,KG
4,-331.59,Qtl
5,119.93,Q
6,187.68,qtl
7,393.67 qtl,NaN
8,14.84,Quintals
9,93.28,qtl


## mapping different crop names refering to one crop

In [9]:
crop_mapping = {
    
    "wheat": "Wheat",
    "गेहूं": "Wheat",
    "gehun": "Wheat",
    "kanak": "Wheat",

    
    "rice": "Rice",
    "paddy": "Rice",
    "chawal": "Rice",
    "चावल": "Rice",
    "धान": "Rice",
    "dhaan": "Rice",
    "basmati": "Rice",

   
    "maize": "Maize",
    "corn": "Maize",
    "मक्का": "Maize",
    "makka": "Maize",
    "makki": "Maize",

  
    "mustard": "Mustard",
    "sarson": "Mustard",
    "sarso": "Mustard",
    "सरसों": "Mustard",


    "cotton": "Cotton",
    "कपास": "Cotton",
    "narma": "Cotton",
    "kapas": "Cotton",

 
    "sugarcane": "Sugarcane",
    "गन्ना": "Sugarcane",
    "ganna": "Sugarcane",
    "ganne": "Sugarcane"
}


crop_key = (
    df_arrivals_clean["crop_name"]
    .astype("string")
    .str.normalize("NFC")
    .str.strip()
    .str.casefold()
)

df_arrivals_clean["crop_name_clean"] = crop_key.map(crop_mapping)

## Checking if every crop name is mapped

In [10]:
unmapped_mask = df_arrivals_clean["crop_name_clean"].isna()

print("Rows with unmapped crops:", int(unmapped_mask.sum()))

if unmapped_mask.any():
    display(
        df_arrivals_clean.loc[unmapped_mask, "crop_name"]
        .value_counts(dropna=False)
        .to_frame("row_count")
    )

print("\nStandardized crop counts:")
display(
    df_arrivals_clean["crop_name_clean"]
    .value_counts()
    .to_frame("row_count")
)

print("\nBefore and after:")
display(
    df_arrivals_clean[
        ["crop_name", "crop_name_clean"]
    ].drop_duplicates().sort_values("crop_name_clean")
)

Rows with unmapped crops: 0

Standardized crop counts:


,row_count
crop_name_clean,
Wheat,4284
Mustard,4209
Sugarcane,4161
Maize,4155
Cotton,4110
Rice,4081



Before and after:


,crop_name,crop_name_clean
61,Narma,Cotton
54,कपास,Cotton
41,Cotton,Cotton
21,cotton,Cotton
6,Kapas,Cotton
22,मक्का,Maize
11,corn,Maize
25,Makka,Maize
73,Makki,Maize
3,Maize,Maize


## Extracting quantites and standardize units

In [11]:
# Preserve the original columns and work with temporary text.
quantity_text = (
    df_arrivals_clean["arrival_quantity"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

unit_text = (
    df_arrivals_clean["unit"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .replace("", pd.NA)
)

# Extract a number and an optional unit from the complete value.
# Examples: "415.88 qtl", "-107.13", "44.841"
quantity_parts = quantity_text.str.extract(
    r"^(?P<number>[+-]?(?:\d+(?:\.\d*)?|\.\d+))"
    r"\s*(?P<embedded_unit>[a-z]+)?$"
)

df_arrivals_clean["quantity_numeric"] = pd.to_numeric(
    quantity_parts["number"],
    errors="coerce"
)

unit_mapping = {
    "q": "Qtl",
    "qtl": "Qtl",
    "quintal": "Qtl",
    "quintals": "Qtl",
    "kg": "KG",
    "kgs": "KG",
    "kilo": "KG",
    "t": "Tonnes",
    "mt": "Tonnes",
    "tonnes": "Tonnes"
}

df_arrivals_clean["unit_from_column"] = unit_text.map(unit_mapping)

df_arrivals_clean["unit_from_quantity"] = (
    quantity_parts["embedded_unit"].map(unit_mapping)
)

display(
    df_arrivals_clean[
        [
            "arrival_quantity",
            "unit",
            "quantity_numeric",
            "unit_from_column",
            "unit_from_quantity"
        ]
    ].head(20)
)

,arrival_quantity,unit,quantity_numeric,unit_from_column,unit_from_quantity
0,44.841,T,44.841,Tonnes,NaN
1,332.54,Qtl,332.54,Qtl,NaN
2,415.88 qtl,NaN,415.88,NaN,Qtl
3,-107.13,KG,-107.13,KG,NaN
4,-331.59,Qtl,-331.59,Qtl,NaN
5,119.93,Q,119.93,Qtl,NaN
6,187.68,qtl,187.68,Qtl,NaN
7,393.67 qtl,NaN,393.67,NaN,Qtl
8,14.84,Quintals,14.84,Qtl,NaN
9,93.28,qtl,93.28,Qtl,NaN


## Combining the unit information 

In [12]:
column_unit = df_arrivals_clean["unit_from_column"]
embedded_unit = df_arrivals_clean["unit_from_quantity"]

# Flag cases where both recognized units exist but disagree.
unit_conflict = (
    column_unit.notna()
    & embedded_unit.notna()
    & column_unit.ne(embedded_unit)
)

# Detect supplied unit labels that our mapping did not recognize.
unknown_unit = (
    (unit_text.notna() & column_unit.isna())
    | (
        quantity_parts["embedded_unit"].notna()
        & embedded_unit.isna()
    )
)

# Use the unit column, or the embedded unit when the column is missing.
df_arrivals_clean["unit_clean"] = column_unit.fillna(embedded_unit)

# Do not choose a unit when the evidence needs review.
df_arrivals_clean.loc[
    unit_conflict | unknown_unit, "unit_clean"
] = pd.NA
# Flagging issues and converting
quantity = df_arrivals_clean["quantity_numeric"]

# Each row receives its first applicable status.
df_arrivals_clean["quantity_status"] = np.select(
    [
        quantity.isna().to_numpy(dtype=bool),
        unit_conflict.fillna(False).to_numpy(dtype=bool),
        unknown_unit.fillna(False).to_numpy(dtype=bool),
        df_arrivals_clean["unit_clean"].isna().to_numpy(dtype=bool),
        quantity.lt(0).fillna(False).to_numpy(dtype=bool)
    ],
    [
        "Unparseable quantity",
        "Conflicting units",
        "Unrecognized unit",
        "Missing unit",
        "Negative quantity"
    ],
    default="Valid"
)

conversion_to_qtl = {
    "Qtl": 1,
    "KG": 0.01,
    "Tonnes": 10
}

valid_quantity = df_arrivals_clean["quantity_status"].eq("Valid")

df_arrivals_clean["arrival_quantity_qtl"] = (
    quantity
    * df_arrivals_clean["unit_clean"].map(conversion_to_qtl)
).where(valid_quantity)
# Review the results of our cleaning and conversion.
display(
    df_arrivals_clean["quantity_status"]
    .value_counts()
    .to_frame("row_count")
)

display(
    df_arrivals_clean[
        [
            "arrival_quantity",
            "unit",
            "unit_clean",
            "arrival_quantity_qtl",
            "quantity_status"
        ]
    ].head(20)
)

,row_count
quantity_status,
Valid,21248
Unparseable quantity,2519
Negative quantity,1233


,arrival_quantity,unit,unit_clean,arrival_quantity_qtl,quantity_status
0,44.841,T,Tonnes,448.41,Valid
1,332.54,Qtl,Qtl,332.54,Valid
2,415.88 qtl,NaN,Qtl,415.88,Valid
3,-107.13,KG,KG,<NA>,Negative quantity
4,-331.59,Qtl,Qtl,<NA>,Negative quantity
5,119.93,Q,Qtl,119.93,Valid
6,187.68,qtl,Qtl,187.68,Valid
7,393.67 qtl,NaN,Qtl,393.67,Valid
8,14.84,Quintals,Qtl,14.84,Valid
9,93.28,qtl,Qtl,93.28,Valid


## Inspecting unparseable values

In [13]:
unparseable_mask = df_arrivals_clean["quantity_status"].eq(
    "Unparseable quantity"
)

print("Sample unparseable quantities:")
display(
    df_arrivals_clean.loc[
        unparseable_mask,
        ["arrival_quantity", "unit"]
    ].head(30)
)

print("\nMost frequent unparseable values:")
display(
    df_arrivals_clean.loc[
        unparseable_mask, "arrival_quantity"
    ]
    .value_counts(dropna=False)
    .head(20)
    .to_frame("row_count")
)

Sample unparseable quantities:


,arrival_quantity,unit
34,"36,654.0 KG",NaN
40,"22,697.0 KG",NaN
65,"43,655.0 KG",NaN
68,"7,278.0 KG",NaN
88,"39,083.0 KG",NaN
91,"35,649.0 KG",NaN
113,"12,646.0 KG",NaN
116,"7,537.0 KG",NaN
117,"41,341.0 KG",NaN
122,"24,318.0 KG",NaN



Most frequent unparseable values:


,row_count
arrival_quantity,
"9,993.0 KG",3
"5,915.0 KG",3
"24,318.0 KG",2
"6,537.0 KG",2
"43,851.0 KG",2
"26,999.0 KG",2
"4,941.0 KG",2
"49,411.0 KG",2
"7,865.000000000001 KG",2


## Repairing unparsable quantites

In [16]:
# Extract comma-formatted KG quantities from the affected rows.
mask = df_arrivals_clean["quantity_status"].eq("Unparseable quantity")

numbers = (
    df_arrivals_clean.loc[mask, "arrival_quantity"]
    .astype("string")
    .str.extract(
        r"(?i)^\s*([+-]?\d{1,3}(?:,\d{3})+(?:\.\d+)?)\s*kg\s*$",
        expand=False
    )
    .str.replace(",", "", regex=False)
)

numbers = pd.to_numeric(numbers, errors="coerce").dropna()

# Repair only rows whose separate unit column was empty.
empty_unit = (
    df_arrivals_clean["unit"].astype("string")
    .fillna("").str.strip().eq("")
)
numbers = numbers.loc[empty_unit.loc[numbers.index]]

df_arrivals_clean.loc[numbers.index, "quantity_numeric"] = numbers
df_arrivals_clean.loc[numbers.index, "unit_from_quantity"] = "KG"
df_arrivals_clean.loc[numbers.index, "unit_clean"] = "KG"

df_arrivals_clean.loc[numbers.index, "arrival_quantity_qtl"] = (
    numbers.div(100).where(numbers.ge(0))
)
df_arrivals_clean.loc[numbers.index, "quantity_status"] = np.where(
    numbers.ge(0), "Valid", "Negative quantity"
)

display(df_arrivals_clean["quantity_status"].value_counts())

quantity_status
Valid                23767
Negative quantity     1233
Name: count, dtype: int64

## Inspecting Mandi's id

In [18]:
print("Distinct mandi ID values:", df_arrivals_clean["mandi_id"].nunique())

print("\nMost frequent mandi ID formats:")
display(
    df_arrivals_clean["mandi_id"]
    .value_counts(dropna=False)
    .head(30)
    .to_frame("row_count")
)

print("\nExamples outside the standard MANDI001 format:")
ids = df_arrivals_clean["mandi_id"].astype("string")

display(
    df_arrivals_clean.loc[
        ~ids.str.fullmatch(r"MANDI\d{3}", na=False),
        ["mandi_id"]
    ].drop_duplicates().head(30)
)

Distinct mandi ID values: 342

Most frequent mandi ID formats:


,row_count
mandi_id,
MANDI002,226
MANDI051,222
MANDI022,221
MANDI048,214
MANDI056,213
MANDI026,212
MANDI020,212
MANDI037,211
MANDI007,210



Examples outside the standard MANDI001 format:


,mandi_id
2,056
4,mandi050
5,mandi_049
6,MANDI-054
8,mandi029
10,mandi_044
12,MANDI-056
14,025
18,MANDI-001
20,mandi_045


## Cleaning Mandi id's

In [20]:
ids = (
    df_arrivals_clean["mandi_id"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.replace(r"[\s_-]+", "", regex=True)
)

numbers = ids.str.extract(
    r"^(?:MANDI|M)?(\d{1,3})$",
    expand=False
)

df_arrivals_clean["mandi_id_clean"] = (
    "MANDI" + numbers.str.zfill(3)
)
clean_ids = df_arrivals_clean["mandi_id_clean"]

df_arrivals_clean["mandi_id_status"] = np.select(
    [
        clean_ids.isna().to_numpy(dtype=bool),
        (~clean_ids.isin(df_master_clean["mandi_id"])).to_numpy(dtype=bool)
    ],
    ["Missing or unrecognized format", "Not found in master"],
    default="Matched"
)

display(
    df_arrivals_clean["mandi_id_status"]
    .value_counts()
    .to_frame("row_count")
)

,row_count
mandi_id_status,
Matched,25000


## Inspecting dates

In [21]:
print("Sample date values:")
display(
    df_arrivals_clean["date"]
    .drop_duplicates()
    .head(30)
    .to_frame()
)

# Replace digits with D to identify the different date structures.
date_patterns = (
    df_arrivals_clean["date"]
    .astype("string")
    .str.strip()
    .str.replace(r"\d", "D", regex=True)
)

print("\nDate formats and counts:")
display(
    date_patterns.value_counts(dropna=False)
    .to_frame("row_count")
)

Sample date values:


,date
0,06-04-2026
1,2026-06-02
2,08-16-2026
3,06-24-2026
4,09.08.2026
5,2026-03-31
6,10/07/2026
7,15/07/2026
8,04-08-2026
9,2026-08-14



Date formats and counts:


,row_count
date,
DDDD-DD-DD,5011
DD/DD/DDDD,4956
DD.DD.DDDD,3805
DDDD/DD/DD,3758
DD-DD-DDDD,3705
DD-Jan-DDDD,485
DD-Mar-DDDD,476
DD-Aug-DDDD,471
DD-Apr-DDDD,467


## Checking if dates are in D/M/Y or M/D/Y formats

In [22]:
dates = df_arrivals_clean["date"].astype("string").str.strip()

for separator in ["/", ".", "-"]:
    parts = dates.str.extract(
        rf"^(\d{{2}})\{separator}(\d{{2}})\{separator}(\d{{4}})$"
    )

    first = pd.to_numeric(parts[0], errors="coerce")
    second = pd.to_numeric(parts[1], errors="coerce")

    print(f"\nSeparator: {separator}")
    print("First number > 12:", int(first.gt(12).sum()))
    print("Second number > 12:", int(second.gt(12).sum()))
    print(
        "Both numbers between 1 and 12:",
        int((first.between(1, 12) & second.between(1, 12)).sum())
    )


Separator: /
First number > 12: 2865
Second number > 12: 0
Both numbers between 1 and 12: 2091

Separator: .
First number > 12: 2278
Second number > 12: 0
Both numbers between 1 and 12: 1527

Separator: -
First number > 12: 0
Second number > 12: 2161
Both numbers between 1 and 12: 1544


## Cleaning dates

In [23]:
dates = df_arrivals_clean["date"].astype("string").str.strip()

formats = {
    r"\d{4}-\d{2}-\d{2}": "%Y-%m-%d",
    r"\d{4}/\d{2}/\d{2}": "%Y/%m/%d",
    r"\d{2}/\d{2}/\d{4}": "%d/%m/%Y",
    r"\d{2}\.\d{2}\.\d{4}": "%d.%m.%Y",
    r"\d{2}-\d{2}-\d{4}": "%m-%d-%Y",
    r"\d{2}-[A-Za-z]{3}-\d{4}": "%d-%b-%Y"
}

df_arrivals_clean["date_clean"] = pd.NaT

for pattern, date_format in formats.items():
    mask = dates.str.fullmatch(pattern, na=False)

    df_arrivals_clean.loc[mask, "date_clean"] = pd.to_datetime(
        dates.loc[mask],
        format=date_format,
        errors="coerce"
    )

# Identify dates where swapping day and month gives another valid date.
parts = dates.str.extract(r"^(\d{2})[-/.](\d{2})[-/.]\d{4}$")
first = pd.to_numeric(parts[0], errors="coerce")
second = pd.to_numeric(parts[1], errors="coerce")

ambiguous = (
    first.between(1, 12)
    & second.between(1, 12)
    & first.ne(second)
)

df_arrivals_clean["date_status"] = "Parsed: unambiguous"

df_arrivals_clean.loc[ambiguous, "date_status"] = (
    "Parsed: assumed format by separator"
)

df_arrivals_clean.loc[
    df_arrivals_clean["date_clean"].isna(), "date_status"
] = "Missing or invalid date"

display(df_arrivals_clean["date_status"].value_counts())

display(
    df_arrivals_clean[
        ["date", "date_clean", "date_status"]
    ].head(15)
)

date_status
Parsed: unambiguous                    20297
Parsed: assumed format by separator     4703
Name: count, dtype: int64

,date,date_clean,date_status
0,06-04-2026,2026-06-04,Parsed: assumed format by separator
1,2026-06-02,2026-06-02,Parsed: unambiguous
2,08-16-2026,2026-08-16,Parsed: unambiguous
3,06-24-2026,2026-06-24,Parsed: unambiguous
4,09.08.2026,2026-08-09,Parsed: assumed format by separator
5,2026-03-31,2026-03-31,Parsed: unambiguous
6,10/07/2026,2026-07-10,Parsed: assumed format by separator
7,15/07/2026,2026-07-15,Parsed: unambiguous
8,04-08-2026,2026-04-08,Parsed: assumed format by separator
9,2026-08-14,2026-08-14,Parsed: unambiguous


## Checking date column

In [24]:
print("Missing or invalid dates:", df_arrivals_clean["date_clean"].isna().sum())
print("Earliest date:", df_arrivals_clean["date_clean"].min())
print("Latest date:", df_arrivals_clean["date_clean"].max())

display(df_arrivals_clean["date_status"].value_counts())
ids = (
    df_arrivals_clean["arrival_id"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
)

df_arrivals_clean["arrival_id_clean"] = ids

repeated = ids.notna() & ids.duplicated(keep=False)

print("Missing arrival IDs:", ids.isna().sum())
print("Rows with repeated nonmissing IDs:", repeated.sum())

display(
    df_arrivals_clean.loc[
        repeated,
        ["arrival_id", "date", "mandi_id", "crop_name", "arrival_quantity"]
    ].sort_values("arrival_id").head(20)
)

Missing or invalid dates: 0
Earliest date: 2026-01-01 00:00:00
Latest date: 2026-09-09 00:00:00


date_status
Parsed: unambiguous                    20297
Parsed: assumed format by separator     4703
Name: count, dtype: int64

Missing arrival IDs: 476
Rows with repeated nonmissing IDs: 0


,arrival_id,date,mandi_id,crop_name,arrival_quantity


## Missing arrival IDs

In [25]:
if "arrival_row_key" not in df_arrivals_clean.columns:
    df_arrivals_clean["arrival_row_key"] = [
        f"ARR_ROW_{i:06d}"
        for i in range(1, len(df_arrivals_clean) + 1)
    ]

assert df_arrivals_clean["arrival_row_key"].is_unique

## Checking variety labels

In [26]:
print("Variety labels and counts:")

display(
    df_arrivals_clean["variety"]
    .value_counts(dropna=False)
    .to_frame("row_count")
)

Variety labels and counts:


,row_count
variety,
NaN,3627
Local,3624
PBW-343,3589
HD-2967,3548
Hybrid,3547
Pusa-1121,3533
Premium,3532


## Cleaning variety column

In [28]:
df_arrivals_clean["variety_clean"] = (
    df_arrivals_clean["variety"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

## Checking farmer's count

In [29]:
raw_farmers = (
    df_arrivals_clean["farmer_count"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

farmers = pd.to_numeric(raw_farmers, errors="coerce")

print("Missing:", raw_farmers.isna().sum())
print("Nonnumeric:", (raw_farmers.notna() & farmers.isna()).sum())
print("Negative:", farmers.lt(0).sum())
print("Zero:", farmers.eq(0).sum())
print(
    "Fractional:",
    (farmers.notna() & farmers.mod(1).ne(0)).sum()
)

Missing: 3800
Nonnumeric: 0
Negative: 0
Zero: 0
Fractional: 0


In [30]:
df_arrivals_clean["farmer_count_clean"] = (
    pd.to_numeric(
        df_arrivals_clean["farmer_count"],
        errors="coerce"
    )
    .astype("Int64")
)

print("Data type:", df_arrivals_clean["farmer_count_clean"].dtype)
print("Missing:", df_arrivals_clean["farmer_count_clean"].isna().sum())

display(
    df_arrivals_clean[
        ["farmer_count", "farmer_count_clean"]
    ].head(10)
)

Data type: Int64
Missing: 3800


,farmer_count,farmer_count_clean
0,128.0,128
1,33.0,33
2,NaN,<NA>
3,112.0,112
4,NaN,<NA>
5,105.0,105
6,45.0,45
7,24.0,24
8,129.0,129
9,109.0,109


## Final validation before exporting

In [31]:
assert len(df_arrivals_clean) == 25000, "Unexpected row count."

assert df_arrivals_clean["arrival_row_key"].notna().all()
assert df_arrivals_clean["arrival_row_key"].is_unique

assert df_arrivals_clean["crop_name_clean"].notna().all()
assert df_arrivals_clean["mandi_id_status"].eq("Matched").all()
assert df_arrivals_clean["date_clean"].notna().all()

valid = df_arrivals_clean["quantity_status"].eq("Valid")
qtl = df_arrivals_clean["arrival_quantity_qtl"]

assert qtl.loc[valid].notna().all(), "Valid quantities are missing."
assert qtl.loc[valid].ge(0).all(), "Negative cleaned quantities found."
assert qtl.loc[~valid].isna().all(), "Invalid quantities entered totals."

assert str(df_arrivals_clean["farmer_count_clean"].dtype) == "Int64"

print("All validation checks passed.")
print("Total records:", len(df_arrivals_clean))
print("Records with valid quantity:", int(valid.sum()))
print("Records excluded from quantity totals:", int((~valid).sum()))
print("Total valid arrivals (quintals):", round(qtl.sum(), 2))

All validation checks passed.
Total records: 25000
Records with valid quantity: 23767
Records excluded from quantity totals: 1233
Total valid arrivals (quintals): 6077460.14


## Exporting files

In [33]:
df_arrivals_clean["arrival_id_status"] = np.where(
    df_arrivals_clean["arrival_id_clean"].isna(),
    "Missing source ID",
    "Present"
)

display(df_arrivals_clean["arrival_id_status"].value_counts())

arrival_id_status
Present              24524
Missing source ID      476
Name: count, dtype: int64

In [34]:
from pathlib import Path

# Final column names mapped to working DataFrame columns.
export_columns = {
    "arrival_row_key": "arrival_row_key",
    "arrival_id": "arrival_id_clean",
    "date": "date_clean",
    "mandi_id": "mandi_id_clean",
    "crop_name": "crop_name_clean",
    "variety": "variety_clean",
    "arrival_quantity_qtl": "arrival_quantity_qtl",
    "farmer_count": "farmer_count_clean",
    "arrival_id_status": "arrival_id_status",
    "date_status": "date_status",
    "mandi_id_status": "mandi_id_status",
    "quantity_status": "quantity_status",
    "source_arrival_id": "arrival_id",
    "source_date": "date",
    "source_mandi_id": "mandi_id",
    "source_crop_name": "crop_name",
    "source_variety": "variety",
    "source_arrival_quantity": "arrival_quantity",
    "source_unit": "unit",
    "source_farmer_count": "farmer_count"
}

arrivals_export = df_arrivals_clean[
    list(export_columns.values())
].copy()

arrivals_export.columns = list(export_columns.keys())

output_folder = Path(
    "/Users/rohankhanna/Documents/datathon/cleaned_data"
)
output_folder.mkdir(parents=True, exist_ok=True)

arrivals_export.to_csv(
    output_folder / "mandi_arrivals_cleaned.csv",
    index=False,
    na_rep="NA",
    date_format="%Y-%m-%d",
    encoding="utf-8-sig"
)

print("Saved:", output_folder / "mandi_arrivals_cleaned.csv")

Saved: /Users/rohankhanna/Documents/datathon/cleaned_data/mandi_arrivals_cleaned.csv
